# Review Metric EDA

This notebook is for exploring the candidate metrics that are most useful for separating likely periodic contaminants from one-off or irregular dipper-like behavior.

It is designed to work directly from a review database (`review.db`) or from a candidate parquet file.


## What this notebook focuses on

The default plots center on the most informative fields for this question:

- Known-periodic metadata: `vetting_likely_known`, `vsx_class`, `asassn_var_type`, `gaia_var_class`, `simbad_otype`
- Catalog period agreement: `catalog_match`, `period_n_sources`, `period_consensus_days`, `period_consensus_agree`
- Native periodicity: `phase_quality_score`, `periodicity_score`, `lsp_is_significant`, `lsp_bootstrap_sig`, `lsp_period`, `lsp_is_alias`
- Recurrence: `dip_is_single_event`, `dip_run_count`, `dipper_n_valid_dips`, `dip_inter_event_spacing_median`, `dip_inter_event_spacing_std`
- Repeatability: `dip_amplitude_consistency`, `dip_duration_consistency`

There are already broader notebooks in `malca/notebooks/` such as `comprehensive_eda.ipynb` and `vetting_stats.ipynb`, but this one is narrower and built around the current review metrics / review DB workflow.


## Click-through mini viewer

If you run the lightweight viewer in a separate terminal, the Plotly scatters in this notebook can open the selected candidate in that viewer when you click a point.

Example:

```bash
malca review-mini --source output/runs/output_bundle_13_13.5_bundle_13_13.5/review/review.db
```

By default the notebook sends clicks to `http://127.0.0.1:8061`. Set `CLICK_VIEWER_URL = None` in the config cell if you want plain Plotly output instead.

In [ ]:
import json
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from IPython.display import HTML, display

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 60)
plt.rcParams['figure.figsize'] = (9, 6)


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start_path = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'pyproject.toml').exists() or (candidate / '.git').exists():
            return candidate
    return start_path


def count_candidates_in_db(db_path: Path) -> int:
    try:
        with sqlite3.connect(db_path) as conn:
            row = conn.execute('SELECT COUNT(*) FROM candidates').fetchone()
            return int(row[0]) if row is not None else -1
    except Exception:
        return -1


def detect_default_source(repo_root: Path) -> tuple[str, Path | None]:
    db_candidates: list[tuple[int, Path]] = []
    for path in sorted((repo_root / 'output' / 'runs').glob('*/review/review.db')):
        count = count_candidates_in_db(path)
        if count >= 0:
            db_candidates.append((count, path))
    for path in [repo_root / 'output' / 'review' / 'review.db', repo_root / 'output' / 'review' / 'standalone.db']:
        if path.exists():
            count = count_candidates_in_db(path)
            if count >= 0:
                db_candidates.append((count, path))
    populated_dbs = [item for item in db_candidates if item[0] > 0]
    if populated_dbs:
        return 'db', max(populated_dbs, key=lambda item: (item[0], str(item[1])))[1]

    parquet_candidates = []
    parquet_patterns = [
        'output/runs/*/results/lc_events_vetted.parquet',
        'output/runs/*/results/lc_events_spectra.parquet',
        'output/runs/*/results/lc_events_neighbors.parquet',
        'output/runs/*/results/lc_events_classified.parquet',
        'output/runs/*/results/lc_events_characterized.parquet',
        'output/runs/*/results/lc_events_filtered.parquet',
    ]
    for pattern in parquet_patterns:
        parquet_candidates.extend(sorted(repo_root.glob(pattern)))
    if parquet_candidates:
        return 'parquet', parquet_candidates[0]
    return 'db', None


REPO_ROOT = find_repo_root()
AUTO_SOURCE_KIND, AUTO_SOURCE_PATH = detect_default_source(REPO_ROOT)

SOURCE_KIND = AUTO_SOURCE_KIND   # 'db' or 'parquet'
SOURCE_PATH = AUTO_SOURCE_PATH   # override if you want a specific DB or parquet
PLOT_SAMPLE = 4000               # sampled point count for scatter plots / pairplots
CLICK_VIEWER_URL = 'http://127.0.0.1:8061'  # set to None to disable click-through mini viewer

BEST_FIELDS = [
    'vetting_likely_known', 'vsx_class', 'asassn_var_type', 'gaia_var_class', 'simbad_otype',
    'catalog_match', 'period_n_sources', 'period_consensus_days', 'period_consensus_agree',
    'phase_quality_score', 'periodicity_score', 'lsp_is_significant', 'lsp_bootstrap_sig', 'lsp_period', 'lsp_is_alias',
    'dip_is_single_event', 'dip_run_count', 'dipper_n_valid_dips', 'dip_inter_event_spacing_median', 'dip_inter_event_spacing_std',
    'dip_amplitude_consistency', 'dip_duration_consistency', 'dipper_score', 'final_class', 'P_eb', 'P_disk', 'P_starspot', 'P_cv',
]

print(f'Repo root: {REPO_ROOT}')
print(f'Source kind: {SOURCE_KIND}')
print(f'Source path: {SOURCE_PATH}')


In [ ]:
def _loads_payload(payload_json: object) -> dict:
    if payload_json in (None, '', b''):
        return {}
    try:
        obj = json.loads(payload_json)
    except Exception:
        return {}
    return obj if isinstance(obj, dict) else {}


def load_review_db(db_path: Path) -> pd.DataFrame:
    db_path = Path(db_path).expanduser().resolve()
    with sqlite3.connect(db_path) as conn:
        candidates = pd.read_sql_query('SELECT * FROM candidates', conn)
        try:
            reviews = pd.read_sql_query('SELECT * FROM reviews', conn)
        except Exception:
            reviews = pd.DataFrame()

    if 'payload_json' in candidates.columns:
        payload_df = pd.json_normalize(candidates['payload_json'].map(_loads_payload))
        payload_df = payload_df.loc[:, ~payload_df.columns.duplicated()].reindex(candidates.index)
        base = candidates.drop(columns=['payload_json'])
        shared_cols = [col for col in payload_df.columns if col in base.columns]
        payload_only = payload_df.drop(columns=shared_cols, errors='ignore')
        if shared_cols:
            shared = base[shared_cols].combine_first(payload_df[shared_cols])
            base = base.drop(columns=shared_cols, errors='ignore')
            candidates = pd.concat([base, shared, payload_only], axis=1)
        else:
            candidates = pd.concat([base, payload_only], axis=1)

    candidates = candidates.loc[:, ~candidates.columns.duplicated()].copy()

    if not reviews.empty and 'candidate_id' in reviews.columns:
        rename_map = {
            col: (f'review_{col}' if col in candidates.columns and col != 'candidate_id' else col)
            for col in reviews.columns
        }
        reviews = reviews.rename(columns=rename_map)
        candidates = candidates.merge(reviews, on='candidate_id', how='left')

    candidates['source_kind'] = 'db'
    candidates['source_file'] = str(db_path)
    return candidates


def load_candidate_parquet(parquet_path: Path) -> pd.DataFrame:
    parquet_path = Path(parquet_path).expanduser().resolve()
    df = pd.read_parquet(parquet_path)
    df = df.copy()
    df['source_kind'] = 'parquet'
    df['source_file'] = str(parquet_path)
    return df


def load_candidates(source_kind: str, source_path: Path | None) -> pd.DataFrame:
    if source_path is None:
        raise FileNotFoundError('No review DB or parquet file was auto-detected. Set SOURCE_PATH manually.')
    if source_kind == 'db':
        return load_review_db(source_path)
    if source_kind == 'parquet':
        return load_candidate_parquet(source_path)
    raise ValueError(f'Unsupported SOURCE_KIND: {source_kind}')


df_raw = load_candidates(SOURCE_KIND, SOURCE_PATH)
print(f'Loaded {len(df_raw):,} rows and {len(df_raw.columns):,} columns')
display(df_raw.head(3))


In [ ]:
TRUE_SET = {'1', 'true', 't', 'yes', 'y'}
FALSE_SET = {'0', 'false', 'f', 'no', 'n'}


def bool_series(frame: pd.DataFrame, col: str, default: bool = False) -> pd.Series:
    if col not in frame.columns:
        return pd.Series(default, index=frame.index, dtype='bool')
    s = frame[col]
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(default)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float) != 0
    text = s.astype(str).str.strip().str.lower()
    out = pd.Series(default, index=frame.index, dtype='bool')
    out.loc[text.isin(TRUE_SET)] = True
    out.loc[text.isin(FALSE_SET)] = False
    return out


def numeric_series(frame: pd.DataFrame, col: str) -> pd.Series:
    if col not in frame.columns:
        return pd.Series(np.nan, index=frame.index, dtype='float64')
    return pd.to_numeric(frame[col], errors='coerce')


def text_series(frame: pd.DataFrame, col: str) -> pd.Series:
    if col not in frame.columns:
        return pd.Series('', index=frame.index, dtype='object')
    return frame[col].fillna('').astype(str)


def contains_periodic_label(series: pd.Series) -> pd.Series:
    patterns = [
        r'\bEA\b', r'\bEB\b', r'\bEW\b', r'\bELL\b', r'\bECL\b',
        r'RR', r'CEP', r'DSCT', r'ROT', r'LPV', r'MIRA', r'ACV', r'BY', r'W UMA', r'ALGOL',
        r'EB\*', r'CV', r'PERIODIC'
    ]
    upper = series.fillna('').astype(str).str.upper()
    mask = pd.Series(False, index=series.index)
    for pattern in patterns:
        mask |= upper.str.contains(pattern, regex=True, na=False)
    return mask


def normalize_review_label(series: pd.Series) -> pd.Series:
    return (
        series.fillna('')
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'unknown dipper': 'dipper'})
    )


def add_eda_columns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    numeric_cols = [
        'period_n_sources', 'period_consensus_days', 'phase_quality_score', 'periodicity_score',
        'lsp_bootstrap_sig', 'lsp_power', 'lsp_period', 'dip_run_count', 'dipper_n_valid_dips',
        'dip_inter_event_spacing_median', 'dip_inter_event_spacing_std',
        'dip_amplitude_consistency', 'dip_duration_consistency', 'dipper_score',
        'P_eb', 'P_disk', 'P_starspot', 'P_cv',
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')

    periodic_text_match = (
        contains_periodic_label(text_series(out, 'vsx_class'))
        | contains_periodic_label(text_series(out, 'asassn_var_type'))
        | contains_periodic_label(text_series(out, 'gaia_var_class'))
        | contains_periodic_label(text_series(out, 'simbad_otype'))
        | contains_periodic_label(text_series(out, 'alerce_lc_class'))
        | contains_periodic_label(text_series(out, 'tns_type'))
    )

    out['known_periodic_catalog'] = bool_series(out, 'vetting_likely_known') | periodic_text_match
    out['strong_catalog_period'] = (
        bool_series(out, 'catalog_match')
        & bool_series(out, 'period_consensus_agree')
        & (numeric_series(out, 'period_n_sources').fillna(0) >= 2)
    )
    out['strong_native_period'] = (
        bool_series(out, 'lsp_is_significant')
        & (~bool_series(out, 'lsp_is_alias'))
        & (numeric_series(out, 'phase_quality_score').fillna(-np.inf) >= 0.5)
    )
    out['recurrent_dips'] = (
        (numeric_series(out, 'dip_run_count').fillna(0) >= 2)
        | (numeric_series(out, 'dipper_n_valid_dips').fillna(0) >= 3)
    )
    out['oneoff_like'] = (
        bool_series(out, 'dip_is_single_event')
        | (numeric_series(out, 'dip_run_count').fillna(0) <= 1)
    )

    out['periodic_evidence_score'] = (
        out['known_periodic_catalog'].astype(int)
        + out['strong_catalog_period'].astype(int)
        + out['strong_native_period'].astype(int)
        + out['recurrent_dips'].astype(int)
    )
    out['periodic_evidence_bucket'] = pd.Categorical(
        np.select(
            [
                out['periodic_evidence_score'] >= 3,
                out['periodic_evidence_score'] == 2,
                out['periodic_evidence_score'] == 1,
            ],
            ['3+ signals', '2 signals', '1 signal'],
            default='0 signals',
        ),
        categories=['0 signals', '1 signal', '2 signals', '3+ signals'],
        ordered=True,
    )

    review_label = normalize_review_label(text_series(out, 'event_class'))
    if 'review_event_class' in out.columns:
        review_label = review_label.where(review_label.ne(''), normalize_review_label(text_series(out, 'review_event_class')))
    out['review_label'] = review_label
    out['is_reviewed'] = review_label.isin({'dipper', 'yso', 'microlensing', 'flare', 'instrumental', 'unknown_interesting', 'other'})
    out['is_reviewed_dipper'] = review_label.eq('dipper')
    out['is_reviewed_non_dipper'] = out['is_reviewed'] & (~out['is_reviewed_dipper'])

    out['proxy_periodic_contaminant'] = out['known_periodic_catalog'] | out['strong_catalog_period']
    out['proxy_oneoff_dipper'] = (
        (~out['proxy_periodic_contaminant'])
        & out['oneoff_like']
        & (numeric_series(out, 'dipper_score').fillna(0) >= 5)
    )

    if out['is_reviewed_dipper'].any():
        out['default_target'] = out['is_reviewed_dipper']
        out['default_reject'] = out['is_reviewed_non_dipper']
        default_target_name = 'is_reviewed_dipper'
        default_reject_name = 'is_reviewed_non_dipper'
    else:
        out['default_target'] = out['proxy_oneoff_dipper']
        out['default_reject'] = out['proxy_periodic_contaminant']
        default_target_name = 'proxy_oneoff_dipper'
        default_reject_name = 'proxy_periodic_contaminant'

    out.attrs['default_target_col'] = default_target_name
    out.attrs['default_reject_col'] = default_reject_name

    out['final_class_label'] = text_series(out, 'final_class').replace('', 'unknown')
    return out


df = add_eda_columns(df_raw)
DEFAULT_TARGET_COL = df.attrs.get('default_target_col', 'default_target')
DEFAULT_REJECT_COL = df.attrs.get('default_reject_col', 'default_reject')
summary = pd.DataFrame(
    {
        'count': [len(df)],
        'reviewed_rows': [int(df['is_reviewed'].sum())],
        'known_periodic_catalog': [int(df['known_periodic_catalog'].sum())],
        'strong_catalog_period': [int(df['strong_catalog_period'].sum())],
        'strong_native_period': [int(df['strong_native_period'].sum())],
        'recurrent_dips': [int(df['recurrent_dips'].sum())],
        'oneoff_like': [int(df['oneoff_like'].sum())],
        'default_target_col': [DEFAULT_TARGET_COL],
        'default_reject_col': [DEFAULT_REJECT_COL],
    }
)
display(summary)


In [ ]:
coverage = pd.DataFrame(
    {
        'field': BEST_FIELDS,
        'present': [field in df.columns for field in BEST_FIELDS],
        'non_null_fraction': [float(df[field].notna().mean()) if field in df.columns else np.nan for field in BEST_FIELDS],
    }
).sort_values(['present', 'non_null_fraction', 'field'], ascending=[False, False, True])

display(coverage)

native_period_fields = ['phase_quality_score', 'periodicity_score', 'lsp_is_significant', 'lsp_bootstrap_sig', 'lsp_period']
native_missing = [field for field in native_period_fields if field in df.columns and float(df[field].notna().mean()) == 0.0]
if native_missing:
    print('Note: these native periodicity fields are entirely empty in the current source:')
    print('  ' + ', '.join(native_missing))
    print('Plots using them will be blank until those stats are refreshed into the review DB/parquet.')


In [ ]:
def sample_frame(frame: pd.DataFrame, max_points: int | None = None) -> pd.DataFrame:
    if max_points is None or len(frame) <= max_points:
        return frame.copy()
    return frame.sample(max_points, random_state=42).copy()


def candidate_table(
    frame: pd.DataFrame,
    columns: list[str],
    *,
    query: str | None = None,
    sort_by: list[str] | str | None = None,
    ascending: bool | list[bool] = False,
    n: int = 25,
) -> pd.DataFrame:
    data = frame.query(query).copy() if query else frame.copy()
    if sort_by is not None:
        data = data.sort_values(sort_by, ascending=ascending)
    cols = [col for col in columns if col in data.columns]
    return data.loc[:, cols].head(n)


def interactive_scatter(
    frame: pd.DataFrame,
    x: str,
    y: str,
    *,
    color: str = 'periodic_evidence_bucket',
    symbol: str | None = None,
    query: str | None = None,
    hover_cols: list[str] | None = None,
    log_x: bool = False,
    log_y: bool = False,
    max_points: int | None = PLOT_SAMPLE,
    identity_line: bool = False,
    title: str | None = None,
    viewer_url: str | None = CLICK_VIEWER_URL,
):
    data = frame.query(query).copy() if query else frame.copy()
    keep = [col for col in [x, y, color, symbol] if col is not None and col in data.columns]
    if not keep:
        raise ValueError('None of the requested columns are present in the dataframe.')
    if x not in data.columns or y not in data.columns:
        missing = [col for col in [x, y] if col not in data.columns]
        raise KeyError(f'Missing columns: {missing}')

    data = data.loc[data[x].notna() & data[y].notna()].copy()
    data = sample_frame(data, max_points)
    if hover_cols is None:
        hover_cols = [
            col for col in [
                'candidate_id', 'asas_sn_id', 'gaia_id', 'final_class', 'review_event_class',
                'vsx_class', 'gaia_var_class', 'asassn_var_type', 'dip_best_morph',
                'period_consensus_days', 'lsp_period', 'dipper_score'
            ] if col in data.columns
        ]

    custom_data = ['candidate_id'] if 'candidate_id' in data.columns else None
    fig = px.scatter(
        data,
        x=x,
        y=y,
        color=color if color in data.columns else None,
        symbol=symbol if symbol in data.columns else None,
        hover_data=hover_cols,
        custom_data=custom_data,
        opacity=0.65,
        title=title or f'{y} vs {x}',
    )
    fig.update_layout(template='plotly_white', height=520)
    fig.update_traces(marker={'size': 8})
    fig.update_xaxes(type='log' if log_x else 'linear')
    fig.update_yaxes(type='log' if log_y else 'linear')

    if identity_line and not data.empty:
        lo = np.nanmin([data[x].min(), data[y].min()])
        hi = np.nanmax([data[x].max(), data[y].max()])
        fig.add_shape(type='line', x0=lo, y0=lo, x1=hi, y1=hi, line={'dash': 'dash', 'color': 'black'})

    if viewer_url and 'candidate_id' in data.columns:
        from malca.review.mini_viewer import clickable_figure_html
        display(HTML(clickable_figure_html(fig, viewer_url=viewer_url, include_plotlyjs='cdn')))
    else:
        fig.show()
    return fig


def compare_distributions(
    frame: pd.DataFrame,
    metrics: list[str],
    *,
    group_col: str = 'periodic_evidence_bucket',
    query: str | None = None,
    bins: int = 40,
):
    data = frame.query(query).copy() if query else frame.copy()
    metrics = [metric for metric in metrics if metric in data.columns]
    if not metrics:
        raise ValueError('None of the requested metrics are present in the dataframe.')

    ncols = 2
    nrows = int(np.ceil(len(metrics) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.5 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, metric in zip(axes, metrics):
        plot_df = data.loc[data[metric].notna()].copy()
        if plot_df.empty:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(metric)
            ax.set_axis_on()
            continue
        hue_col = None
        if group_col in plot_df.columns and plot_df[group_col].notna().any():
            hue_col = group_col
        sns.histplot(
            data=plot_df,
            x=metric,
            hue=hue_col,
            bins=bins,
            element='step',
            stat='density',
            common_norm=False,
            ax=ax,
        )
        ax.set_title(metric)
    for ax in axes[len(metrics):]:
        ax.set_axis_off()
    plt.tight_layout()


In [ ]:
interactive_scatter(
    df,
    x='phase_quality_score',
    y='periodicity_score',
    color='periodic_evidence_bucket',
    symbol='dip_is_single_event',
    title='Native periodicity: phase quality vs periodicity score',
)

interactive_scatter(
    df,
    x='period_consensus_days',
    y='lsp_period',
    color='periodic_evidence_bucket',
    log_x=True,
    log_y=True,
    identity_line=True,
    title='Catalog period vs native LSP period',
)

interactive_scatter(
    df,
    x='dip_run_count',
    y='dipper_n_valid_dips',
    color='known_periodic_catalog',
    symbol='dip_is_single_event',
    title='Recurrence: dip runs vs valid dips',
)

interactive_scatter(
    df,
    x='dip_inter_event_spacing_median',
    y='dip_inter_event_spacing_std',
    color='periodic_evidence_bucket',
    log_x=True,
    log_y=True,
    title='Recurrence regularity: spacing median vs spacing scatter',
)

interactive_scatter(
    df,
    x='dip_amplitude_consistency',
    y='dip_duration_consistency',
    color='periodic_evidence_bucket',
    title='Repeatability: amplitude consistency vs duration consistency',
)


In [ ]:
compare_distributions(
    df,
    metrics=[
        'period_n_sources', 'phase_quality_score', 'periodicity_score',
        'dip_run_count', 'dipper_n_valid_dips', 'lsp_bootstrap_sig',
    ],
    group_col='periodic_evidence_bucket',
)


In [ ]:
candidate_cols = [
    'candidate_id', 'asas_sn_id', 'gaia_id', 'final_class', 'review_event_class',
    'known_periodic_catalog', 'strong_catalog_period', 'strong_native_period', 'recurrent_dips', 'oneoff_like',
    'period_n_sources', 'period_consensus_days', 'period_consensus_agree',
    'phase_quality_score', 'periodicity_score', 'lsp_is_significant', 'lsp_is_alias', 'lsp_period',
    'dip_is_single_event', 'dip_run_count', 'dipper_n_valid_dips',
    'dip_inter_event_spacing_median', 'dip_inter_event_spacing_std',
    'dip_amplitude_consistency', 'dip_duration_consistency', 'dipper_score',
    'vsx_class', 'gaia_var_class', 'asassn_var_type', 'simbad_otype',
]

print('Likely periodic contaminants or repeaters:')
display(
    candidate_table(
        df,
        candidate_cols,
        query='known_periodic_catalog or strong_catalog_period or strong_native_period or recurrent_dips',
        sort_by=['periodic_evidence_score', 'phase_quality_score', 'periodicity_score', 'dipper_score'],
        ascending=[False, False, False, False],
        n=25,
    )
)

print('One-off-like dippers without strong catalog evidence:')
display(
    candidate_table(
        df,
        candidate_cols,
        query='(~known_periodic_catalog) and oneoff_like',
        sort_by=['dipper_score', 'phase_quality_score', 'periodicity_score'],
        ascending=[False, False, False],
        n=25,
    )
)


## Cut exploration and box cuts

A useful next step is to move beyond scatter plots and ask: *which simple cuts separate one-off dipper-like objects from periodic contaminants?*

The cells below add:

- label-aware cut scoring when you have reviewed labels
- proxy targets when you do not
- 1D threshold sweeps
- 2D box-cut sweeps
- a scatter helper that overlays the cut region

This is useful for exploring simple triage rules like:

- `period_n_sources >= 2`
- `known_periodic_catalog == True`
- `dip_run_count <= 1 and period_n_sources <= 1`
- `dip_run_count >= 2 and dip_inter_event_spacing_std / dip_inter_event_spacing_median < x`


In [ ]:
def query_mask(frame: pd.DataFrame, query: str | None = None) -> pd.Series:
    if not query:
        return pd.Series(True, index=frame.index, dtype='bool')
    idx = frame.query(query, engine='python').index
    return pd.Series(frame.index.isin(idx), index=frame.index, dtype='bool')


def cut_summary(
    frame: pd.DataFrame,
    mask: pd.Series,
    *,
    target_col: str = DEFAULT_TARGET_COL,
    reject_col: str = DEFAULT_REJECT_COL,
    eligible_query: str | None = None,
    name: str = 'cut',
) -> pd.Series:
    eligible = query_mask(frame, eligible_query)
    selected = eligible & mask.fillna(False).astype(bool)
    target = eligible & bool_series(frame, target_col)
    reject = eligible & bool_series(frame, reject_col)

    n_selected = int(selected.sum())
    n_target = int(target.sum())
    n_reject = int(reject.sum())
    n_selected_target = int((selected & target).sum())
    n_selected_reject = int((selected & reject).sum())

    purity = n_selected_target / n_selected if n_selected else np.nan
    target_recall = n_selected_target / n_target if n_target else np.nan
    reject_leakage = n_selected_reject / n_selected if n_selected else np.nan
    lift = purity / (n_target / int(eligible.sum())) if n_selected and int(eligible.sum()) and n_target else np.nan

    return pd.Series({
        'name': name,
        'target_col': target_col,
        'reject_col': reject_col,
        'eligible': int(eligible.sum()),
        'selected': n_selected,
        'target_total': n_target,
        'reject_total': n_reject,
        'selected_target': n_selected_target,
        'selected_reject': n_selected_reject,
        'purity': purity,
        'target_recall': target_recall,
        'reject_leakage': reject_leakage,
        'lift_vs_base_rate': lift,
    })


def evaluate_query(
    frame: pd.DataFrame,
    query: str,
    *,
    target_col: str = DEFAULT_TARGET_COL,
    reject_col: str = DEFAULT_REJECT_COL,
    eligible_query: str | None = None,
) -> pd.Series:
    return cut_summary(
        frame,
        query_mask(frame, query),
        target_col=target_col,
        reject_col=reject_col,
        eligible_query=eligible_query,
        name=query,
    )


def sweep_threshold(
    frame: pd.DataFrame,
    metric: str,
    *,
    op: str = '>=',
    thresholds: np.ndarray | list[float] | None = None,
    target_col: str = DEFAULT_TARGET_COL,
    reject_col: str = DEFAULT_REJECT_COL,
    eligible_query: str | None = None,
) -> pd.DataFrame:
    values = numeric_series(frame, metric)
    eligible = query_mask(frame, eligible_query) & values.notna()
    valid = values.loc[eligible]
    if valid.empty:
        return pd.DataFrame()
    if thresholds is None:
        thresholds = np.unique(np.nanquantile(valid, np.linspace(0.05, 0.95, 19)))

    rows = []
    for threshold in thresholds:
        if op == '>=':
            mask = values >= threshold
        elif op == '>':
            mask = values > threshold
        elif op == '<=':
            mask = values <= threshold
        elif op == '<':
            mask = values < threshold
        else:
            raise ValueError(f'Unsupported operator: {op}')
        row = cut_summary(
            frame,
            mask,
            target_col=target_col,
            reject_col=reject_col,
            eligible_query=eligible_query,
            name=f'{metric} {op} {threshold:g}',
        )
        row['metric'] = metric
        row['op'] = op
        row['threshold'] = float(threshold)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(['purity', 'target_recall', 'selected'], ascending=[False, False, False])


def sweep_2d_box(
    frame: pd.DataFrame,
    x: str,
    y: str,
    *,
    x_op: str = '>=',
    y_op: str = '>=',
    x_thresholds: np.ndarray | list[float] | None = None,
    y_thresholds: np.ndarray | list[float] | None = None,
    target_col: str = DEFAULT_TARGET_COL,
    reject_col: str = DEFAULT_REJECT_COL,
    eligible_query: str | None = None,
) -> pd.DataFrame:
    xvals = numeric_series(frame, x)
    yvals = numeric_series(frame, y)
    eligible = query_mask(frame, eligible_query) & xvals.notna() & yvals.notna()
    xvalid = xvals.loc[eligible]
    yvalid = yvals.loc[eligible]
    if xvalid.empty or yvalid.empty:
        return pd.DataFrame()

    if x_thresholds is None:
        x_thresholds = np.unique(np.nanquantile(xvalid, np.linspace(0.1, 0.9, 9)))
    if y_thresholds is None:
        y_thresholds = np.unique(np.nanquantile(yvalid, np.linspace(0.1, 0.9, 9)))

    def apply_op(series: pd.Series, threshold: float, op: str) -> pd.Series:
        if op == '>=':
            return series >= threshold
        if op == '>':
            return series > threshold
        if op == '<=':
            return series <= threshold
        if op == '<':
            return series < threshold
        raise ValueError(f'Unsupported operator: {op}')

    rows = []
    for xt in x_thresholds:
        for yt in y_thresholds:
            mask = apply_op(xvals, float(xt), x_op) & apply_op(yvals, float(yt), y_op)
            row = cut_summary(
                frame,
                mask,
                target_col=target_col,
                reject_col=reject_col,
                eligible_query=eligible_query,
                name=f'{x} {x_op} {xt:g} and {y} {y_op} {yt:g}',
            )
            row['x'] = x
            row['x_op'] = x_op
            row['x_threshold'] = float(xt)
            row['y'] = y
            row['y_op'] = y_op
            row['y_threshold'] = float(yt)
            rows.append(row)
    return pd.DataFrame(rows).sort_values(['purity', 'target_recall', 'selected'], ascending=[False, False, False])


def plot_box_cut(
    frame: pd.DataFrame,
    *,
    x: str,
    y: str,
    x_min: float | None = None,
    x_max: float | None = None,
    y_min: float | None = None,
    y_max: float | None = None,
    color: str = 'periodic_evidence_bucket',
    query: str | None = None,
    max_points: int | None = PLOT_SAMPLE,
    title: str | None = None,
):
    fig = interactive_scatter(
        frame,
        x=x,
        y=y,
        color=color,
        query=query,
        max_points=max_points,
        title=title or f'Box cut: {y} vs {x}',
    )
    data = frame.query(query).copy() if query else frame.copy()
    data = data.loc[data[x].notna() & data[y].notna()]
    if data.empty:
        return fig

    x0 = x_min if x_min is not None else float(data[x].min())
    x1 = x_max if x_max is not None else float(data[x].max())
    y0 = y_min if y_min is not None else float(data[y].min())
    y1 = y_max if y_max is not None else float(data[y].max())
    fig.add_shape(
        type='rect',
        x0=x0, x1=x1, y0=y0, y1=y1,
        line={'color': 'black', 'width': 2},
        fillcolor='rgba(0, 0, 0, 0)',
    )
    fig.show()
    return fig


In [ ]:
print(f'Default cut target: {DEFAULT_TARGET_COL}')
print(f'Default cut reject: {DEFAULT_REJECT_COL}')

example_cuts = pd.DataFrame([
    evaluate_query(df, '(known_periodic_catalog) or (period_n_sources >= 2)', target_col='proxy_periodic_contaminant', reject_col='proxy_oneoff_dipper', eligible_query='dipper_score >= 5'),
    evaluate_query(df, '(~known_periodic_catalog) and (period_n_sources <= 1) and oneoff_like', target_col='proxy_oneoff_dipper', reject_col='proxy_periodic_contaminant', eligible_query='dipper_score >= 5'),
    evaluate_query(df, '(dip_run_count >= 2) and (period_n_sources >= 2)', target_col='proxy_periodic_contaminant', reject_col='proxy_oneoff_dipper', eligible_query='dipper_score >= 5'),
    evaluate_query(df, '(dip_run_count <= 1) and (dipper_n_valid_dips >= 10)', target_col='proxy_oneoff_dipper', reject_col='proxy_periodic_contaminant', eligible_query='dipper_score >= 5'),
])
display(example_cuts)


In [ ]:
periodic_box_sweep = sweep_2d_box(
    df,
    x='period_n_sources',
    y='dip_run_count',
    x_op='>=',
    y_op='>=',
    target_col='proxy_periodic_contaminant',
    reject_col='proxy_oneoff_dipper',
    eligible_query='dipper_score >= 5',
)

oneoff_box_sweep = sweep_2d_box(
    df,
    x='period_n_sources',
    y='dip_run_count',
    x_op='<=',
    y_op='<=',
    target_col='proxy_oneoff_dipper',
    reject_col='proxy_periodic_contaminant',
    eligible_query='dipper_score >= 5',
)

print('Top periodic-contaminant-style box cuts:')
display(periodic_box_sweep.head(15))

print('Top one-off-dipper-style box cuts:')
display(oneoff_box_sweep.head(15))


In [ ]:
_ = plot_box_cut(
    df,
    x='period_n_sources',
    y='dip_run_count',
    x_min=2,
    y_min=2,
    color='proxy_periodic_contaminant',
    query='dipper_score >= 5',
    title='Simple periodic-contaminant box cut',
)

_ = plot_box_cut(
    df,
    x='period_n_sources',
    y='dip_run_count',
    x_max=1,
    y_max=1,
    color='proxy_oneoff_dipper',
    query='dipper_score >= 5',
    title='Simple one-off-dipper box cut',
)


In [ ]:
# Optional custom exploration examples.
#
# interactive_scatter(
#     df,
#     x='phase_quality_score',
#     y='dipper_score',
#     color='gaia_var_class',
#     query='dipper_score >= 5',
#     max_points=3000,
# )
#
# interactive_scatter(
#     df,
#     x='periodicity_score',
#     y='dip_inter_event_spacing_std',
#     color='final_class_label',
#     query='dip_run_count >= 2',
#     log_y=True,
# )
#
# display(candidate_table(
#     df,
#     candidate_cols,
#     query='(~known_periodic_catalog) and strong_native_period and recurrent_dips',
#     sort_by=['phase_quality_score', 'periodicity_score'],
#     ascending=[False, False],
#     n=50,
# ))
